# Clase 9 · Práctica: redes generativas antagónicas (GAN)

En esta práctica completas por tu cuenta el código de una red generativa antagónica (GAN, del inglés *Generative Adversarial Network*), la arquitectura que viste en la Clase 9 del curso "Advanced Deep Learning".

## Qué vas a practicar

- Construyes las dos redes que forman una GAN: el generador y el discriminador, con capas lineales en PyTorch
- Completas el bucle de entrenamiento: la actualización del discriminador y la actualización del generador con la pérdida no saturante
- Observas en un gráfico si el generador aprende a cubrir toda la distribución de datos reales o si cae en un colapso de modo

## Requisito previo

Antes de empezar, asegúrate de haber asistido a la Clase 9 en directo (o de haber revisado su grabación) y de haber leído el capítulo del manual sobre GAN. Esta práctica da por hecho que ya conoces el objetivo minimax, el papel del generador y del discriminador, y por qué se usa la pérdida no saturante.

Cada ejercicio trae el andamiaje de código ya escrito. Completa únicamente las partes marcadas con `# TODO`. Debajo de cada ejercicio tienes una solución colapsada: intenta resolverlo por tu cuenta antes de abrirla.

## Ejercicio 1 · El generador y el discriminador

### Caso

Trabajas en un proyecto con un conjunto de datos muy pequeño: solo tienes un puñado de mediciones de dos sensores que, al representarlas en un plano, forman dos grupos bien diferenciados (una mezcla de dos gaussianas). Quieres entrenar una GAN sobre estos datos para generar ejemplos sintéticos adicionales y así aumentar el conjunto de entrenamiento antes de usarlo en otro modelo.

### Qué falta

La celda siguiente ya trae escritos:

- La función `generar_datos_reales`, que simula tu conjunto de datos pequeño: dos gaussianas centradas en `(-2.5, 0)` y `(2.5, 0)`
- El método `forward` de ambas clases

Te falta completar la definición de las capas de `Generador` y `Discriminador` dentro de `nn.Sequential`.

### Pistas

- El generador recibe un vector de ruido de `latent_dim` dimensiones y debe devolver un punto en dos dimensiones: la última capa lineal tiene que terminar en 2 salidas
- El discriminador recibe un punto en dos dimensiones y devuelve un único logit (no apliques sigmoide: la función de pérdida que usarás en el Ejercicio 2 ya la incluye)
- Usa `nn.LeakyReLU(0.2)` como activación entre capas lineales, igual que en el ejemplo de la Clase 9
- Dos capas ocultas de tamaño `hidden_dim` bastan para este problema tan simple

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Datos reales: mezcla de dos gaussianas separadas en el plano (tu dataset pequeño)
def generar_datos_reales(n_muestras, centros=((-2.5, 0.0), (2.5, 0.0)), std=0.4):
    centros = np.array(centros)
    indices = np.random.randint(0, len(centros), size=n_muestras)
    puntos = centros[indices] + std * np.random.randn(n_muestras, 2)
    return torch.tensor(puntos, dtype=torch.float32)

LATENT_DIM = 2

In [ ]:
class Generador(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, hidden_dim=64):
        super().__init__()
        self.red = nn.Sequential(
            # TODO: define las capas del generador.
            # Estructura esperada: Linear(latent_dim -> hidden_dim), LeakyReLU(0.2),
            # Linear(hidden_dim -> hidden_dim), LeakyReLU(0.2), Linear(hidden_dim -> 2)
            ...
        )

    def forward(self, z):
        return self.red(z)


class Discriminador(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.red = nn.Sequential(
            # TODO: define las capas del discriminador.
            # Estructura esperada: Linear(2 -> hidden_dim), LeakyReLU(0.2),
            # Linear(hidden_dim -> hidden_dim), LeakyReLU(0.2), Linear(hidden_dim -> 1)
            # La última capa no lleva activación: devuelve un logit
            ...
        )

    def forward(self, x):
        return self.red(x)


generador = Generador().to(device)
discriminador = Discriminador().to(device)

print(generador)
print(discriminador)

<details>
<summary>💡 Ver solución</summary>

```python
class Generador(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, hidden_dim=64):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, z):
        return self.red(z)


class Discriminador(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),  # logit, sin sigmoide
        )

    def forward(self, x):
        return self.red(x)


generador = Generador().to(device)
discriminador = Discriminador().to(device)

print(generador)
print(discriminador)
```
</details>

## Ejercicio 2 · El bucle de entrenamiento

### Caso

Ya tienes las dos redes definidas. Ahora te falta la parte que las hace competir: el bucle de entrenamiento que alterna la actualización del discriminador y la del generador, tal como viste en la Clase 9.

### Qué falta

La celda siguiente ya trae escritos:

- La función de pérdida `criterion` (`BCEWithLogitsLoss`, que combina sigmoide y entropía cruzada binaria)
- Los optimizadores `opt_g` y `opt_d`
- El muestreo de datos reales, de ruido y de datos falsos en cada paso
- Las etiquetas `etiquetas_reales` y `etiquetas_falsas`
- El resto del bucle: `zero_grad`, `backward`, `step` e impresión de progreso

Te falta completar dos bloques:

1. **Actualización del discriminador**: calcula `loss_d_real` (la pérdida del discriminador sobre los datos reales, comparados con `etiquetas_reales`), `loss_d_fake` (la pérdida sobre los datos falsos, comparados con `etiquetas_falsas`) y súmalas en `loss_d`
2. **Actualización del generador**: calcula `loss_g` con la pérdida no saturante, es decir, comparando la salida del discriminador sobre `datos_falsos` con `etiquetas_reales` (el generador quiere engañar al discriminador, no que lo detecte)

### Pistas

- `criterion` recibe primero las predicciones del discriminador y después las etiquetas: `criterion(discriminador(...), etiquetas)`
- En el paso del discriminador, los datos falsos se generan con `.detach()`, ya escrito para ti: así el generador no acumula gradiente en esa actualización
- En el paso del generador, vuelve a muestrear ruido y a generar datos falsos sin `.detach()`, porque esta vez sí necesitas que el gradiente llegue hasta el generador
- Recuerda el porqué de la pérdida no saturante: minimizar `log(1 - D(G(z)))` da gradientes casi planos al principio, así que en la práctica se maximiza `log(D(G(z)))`, que en código equivale a comparar contra `etiquetas_reales`

In [ ]:
criterion = nn.BCEWithLogitsLoss()
opt_g = torch.optim.Adam(generador.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(discriminador.parameters(), lr=2e-4, betas=(0.5, 0.999))

BATCH_SIZE = 256
N_STEPS = 3000

for paso in range(1, N_STEPS + 1):
    # ---------- 1) Actualiza el discriminador ----------
    datos_reales = generar_datos_reales(BATCH_SIZE).to(device)
    z = torch.randn(BATCH_SIZE, LATENT_DIM, device=device)
    datos_falsos = generador(z).detach()  # detach: el generador no se actualiza en este paso

    etiquetas_reales = torch.ones(BATCH_SIZE, 1, device=device)
    etiquetas_falsas = torch.zeros(BATCH_SIZE, 1, device=device)

    # TODO: calcula loss_d_real (pérdida sobre datos_reales frente a etiquetas_reales),
    # loss_d_fake (pérdida sobre datos_falsos frente a etiquetas_falsas) y súmalas en loss_d
    loss_d_real = ...
    loss_d_fake = ...
    loss_d = ...

    opt_d.zero_grad()
    loss_d.backward()
    opt_d.step()

    # ---------- 2) Actualiza el generador (pérdida no saturante) ----------
    z = torch.randn(BATCH_SIZE, LATENT_DIM, device=device)
    datos_falsos = generador(z)

    # TODO: calcula loss_g comparando discriminador(datos_falsos) con etiquetas_reales
    loss_g = ...

    opt_g.zero_grad()
    loss_g.backward()
    opt_g.step()

    if paso % 500 == 0:
        print(f"paso {paso:5d} | pérdida D: {loss_d.item():.3f} | pérdida G: {loss_g.item():.3f}")

In [ ]:
# Visualización: datos reales frente a muestras generadas, con la GAN ya entrenada
with torch.no_grad():
    muestra_real = generar_datos_reales(1000).numpy()
    z_vis = torch.randn(1000, LATENT_DIM, device=device)
    muestra_generada = generador(z_vis).cpu().numpy()

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(muestra_real[:, 0], muestra_real[:, 1], s=10, alpha=0.5, label="Datos reales")
ax.scatter(muestra_generada[:, 0], muestra_generada[:, 1], s=10, alpha=0.5, label="Muestras generadas")
ax.set_title("GAN entrenada: datos reales frente a datos generados")
ax.legend()
ax.set_aspect("equal")
plt.show()

<details>
<summary>💡 Ver solución</summary>

```python
criterion = nn.BCEWithLogitsLoss()
opt_g = torch.optim.Adam(generador.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(discriminador.parameters(), lr=2e-4, betas=(0.5, 0.999))

BATCH_SIZE = 256
N_STEPS = 3000

for paso in range(1, N_STEPS + 1):
    # ---------- 1) Actualiza el discriminador ----------
    datos_reales = generar_datos_reales(BATCH_SIZE).to(device)
    z = torch.randn(BATCH_SIZE, LATENT_DIM, device=device)
    datos_falsos = generador(z).detach()  # detach: el generador no se actualiza en este paso

    etiquetas_reales = torch.ones(BATCH_SIZE, 1, device=device)
    etiquetas_falsas = torch.zeros(BATCH_SIZE, 1, device=device)

    # Pérdida del discriminador: bien clasificando reales, bien clasificando falsos
    loss_d_real = criterion(discriminador(datos_reales), etiquetas_reales)
    loss_d_fake = criterion(discriminador(datos_falsos), etiquetas_falsas)
    loss_d = loss_d_real + loss_d_fake

    opt_d.zero_grad()
    loss_d.backward()
    opt_d.step()

    # ---------- 2) Actualiza el generador (pérdida no saturante) ----------
    z = torch.randn(BATCH_SIZE, LATENT_DIM, device=device)
    datos_falsos = generador(z)

    # El generador "quiere" que el discriminador clasifique sus muestras como reales
    loss_g = criterion(discriminador(datos_falsos), etiquetas_reales)

    opt_g.zero_grad()
    loss_g.backward()
    opt_g.step()

    if paso % 500 == 0:
        print(f"paso {paso:5d} | pérdida D: {loss_d.item():.3f} | pérdida G: {loss_g.item():.3f}")
```
</details>

## Para cerrar

Ya completaste una GAN entera: definiste el generador y el discriminador, y programaste el juego minimax que los enfrenta paso a paso. Fíjate en el gráfico final: si el generador solo cubre una de las dos gaussianas, acabas de presenciar en primera persona un colapso de modo, el mismo problema que viste en la Clase 9.

Antes de cerrar el notebook, vuelve a `N_STEPS` y pruébalo con un valor mucho más bajo, como 300. Compara el resultado con tu ejecución anterior y comprueba cuánto entrenamiento necesita esta GAN tan pequeña para cubrir los dos modos de la distribución.

**Comparte tu gráfico final en el canal de la clase y cuenta si tu generador sufrió colapso de modo.**